# Carbon Footprint Model -- Manager Notebook

Orchestrates training of the LUPI-Enhanced Multi-Encoder Carbon Footprint Network.
All model logic lives in `src/` -- this notebook only imports and calls.

## 1. Initialization

In [ ]:
# -- Configuration --
GITHUB_USERNAME = ""  # Fill in
GITHUB_TOKEN = ""     # Fill in (fine-grained PAT)
REPO_URL = ""         # Fill in

import os
import subprocess

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/ESPResso-V2"
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Drive mounted at {DRIVE_BASE}")
else:
    DRIVE_BASE = None
    print("Running locally")

# -- Preflight validation --
errors = []
if IN_COLAB:
    if not GITHUB_USERNAME:
        errors.append("GITHUB_USERNAME is empty -- fill in cell above")
    if not GITHUB_TOKEN:
        errors.append("GITHUB_TOKEN is empty -- fill in cell above")
    if not REPO_URL:
        errors.append("REPO_URL is empty -- fill in cell above")

for pkg in ["torch", "pandas", "numpy", "sklearn"]:
    try:
        __import__(pkg)
    except ImportError:
        errors.append(f"Required package '{pkg}' not importable")

if IN_COLAB:
    lfs_check = subprocess.run(["git", "lfs", "version"], capture_output=True)
    if lfs_check.returncode != 0:
        errors.append("git-lfs not installed. Run: apt-get install git-lfs")

assert not errors, "Preflight failed:\n  " + "\n  ".join(errors)
print("Preflight passed")

## 2. Clone / Update Repository

In [ ]:
if IN_COLAB:
    REPO_DIR = "/content/ESPResso-V2"
    if not os.path.exists(REPO_DIR):
        auth_url = REPO_URL.replace(
            "https://", f"https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@"
        )
        subprocess.run(
            ["git", "clone", auth_url, REPO_DIR],
            check=True,
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
        )
    else:
        subprocess.run(["git", "pull"], check=True, cwd=REPO_DIR)
    os.chdir(REPO_DIR)
else:
    # Local: walk up to repo root from notebook location
    REPO_DIR = os.path.abspath(
        os.path.join(os.path.dirname("__file__"), "..", "..", "..")
    )
    if not os.path.exists(os.path.join(REPO_DIR, "model")):
        REPO_DIR = os.getcwd()

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Repo dir on sys.path: {REPO_DIR}")

## 3. Check Data Cache

In [ ]:
def _is_lfs_pointer(path):
    """True if file is a Git LFS pointer stub, not actual data."""
    if not os.path.exists(path):
        return False
    if os.path.getsize(path) > 1024:
        return False
    with open(path, "r") as f:
        return f.readline().startswith("version https://git-lfs")


DATA_FILE = "carbon_footprint.parquet"
DATA_PATH = os.path.join(REPO_DIR, "data", "training", DATA_FILE)
data_ready = os.path.exists(DATA_PATH) and not _is_lfs_pointer(DATA_PATH)

if IN_COLAB:
    CACHE_DIR = os.path.join(DRIVE_BASE, "data")
    CACHE_PATH = os.path.join(CACHE_DIR, DATA_FILE)

    # Try restoring from Drive cache first
    if not data_ready and os.path.exists(CACHE_PATH) and not _is_lfs_pointer(CACHE_PATH):
        os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
        import shutil
        shutil.copy2(CACHE_PATH, DATA_PATH)
        data_ready = True
        print("Restored data from Drive cache")

    # Fall back to git lfs pull
    if not data_ready:
        print("Pulling data via Git LFS...")
        subprocess.run(
            ["git", "lfs", "pull", "--include=data/training/**"],
            check=True,
            cwd=REPO_DIR,
        )
        data_ready = os.path.exists(DATA_PATH) and not _is_lfs_pointer(DATA_PATH)
        if data_ready:
            os.makedirs(CACHE_DIR, exist_ok=True)
            import shutil
            shutil.copy2(DATA_PATH, CACHE_PATH)
            print("Cached data to Drive")

    # Validate VERSION freshness
    version_path = os.path.join(REPO_DIR, "data", "training", "VERSION")
    cached_version = os.path.join(CACHE_DIR, "VERSION")
    if os.path.exists(version_path):
        with open(version_path) as f:
            repo_version = f.read().strip()
        if os.path.exists(cached_version):
            with open(cached_version) as f:
                cache_version = f.read().strip()
            if repo_version != cache_version:
                print(f"VERSION mismatch (repo={repo_version}, cache={cache_version})")
                print("Re-pulling data via Git LFS...")
                subprocess.run(
                    ["git", "lfs", "pull", "--include=data/training/**"],
                    check=True,
                    cwd=REPO_DIR,
                )
                import shutil
                shutil.copy2(DATA_PATH, CACHE_PATH)
                shutil.copy2(version_path, cached_version)
        else:
            import shutil
            shutil.copy2(version_path, cached_version)

assert data_ready, f"Data not found or still LFS pointer at {DATA_PATH}"
print(f"Data ready: {DATA_PATH} ({os.path.getsize(DATA_PATH) / 1e6:.1f} MB)")

## 4. Imports

In [ ]:
import copy
import glob

import numpy as np
import torch
from torch.utils.data import DataLoader

from model.carbon_footprint.src.utils.config import CarbonConfig, set_seed
from model.carbon_footprint.src.preprocessing.dataset import CarbonDataset
from model.carbon_footprint.src.preprocessing.transforms import (
    HEAD_NAMES, Log1pZScoreTransform, create_splits,
)
from model.carbon_footprint.src.training.model import CarbonModel
from model.carbon_footprint.src.training.loss import ThreeGroupLoss
from model.carbon_footprint.src.training.trainer import CarbonTrainer
from model.carbon_footprint.src.training.checkpoint import smoke_test
from model.carbon_footprint.src.evaluation.metrics import (
    compute_metrics, per_tier_evaluation,
)
from model.carbon_footprint.src.evaluation.plots import (
    setup_style, plot_training_curves, plot_predictions, plot_residuals,
)
from model.carbon_footprint.src.evaluation.plots_extra import (
    plot_tier_degradation, plot_tier_heatmap, plot_loss_groups,
)

print("All imports successful")

## 5. Configuration

In [ ]:
# Choose preset: smoke() for testing, dev() for search, full() for baseline, production() for final
config = CarbonConfig.production()
config.data_dir = os.path.join(REPO_DIR, "data", "training")

if IN_COLAB:
    config.checkpoint_dir = os.path.join(
        DRIVE_BASE, "checkpoints", "carbon_footprint"
    )
    config.num_workers = 4
else:
    config.checkpoint_dir = os.path.join(
        REPO_DIR, "checkpoints", "carbon_footprint"
    )
    config.num_workers = 0
    config.pin_memory = False
    config.persistent_workers = False

device = config.device
set_seed(config.seed)

print(f"Device: {device}")
print(f"Data: {config.data_path}")
print(f"Checkpoints: {config.checkpoint_dir}")
print(f"Preset: full (lr={config.lr}, bs={config.batch_size}, "
      f"trunk={config.trunk_hidden}x{config.trunk_blocks})")

## 5b. Checkpoint Reset

In [ ]:
RESET_CHECKPOINTS = False  # Set to True to clear all checkpoints and start fresh

if RESET_CHECKPOINTS:
    import shutil
    if os.path.exists(config.checkpoint_dir):
        shutil.rmtree(config.checkpoint_dir)
        os.makedirs(config.checkpoint_dir, exist_ok=True)
        print(f"Cleared checkpoints at {config.checkpoint_dir}")
    else:
        print("No checkpoint directory found -- nothing to clear")
else:
    print(f"Checkpoints preserved at {config.checkpoint_dir}")

## 6. Data Loading

In [ ]:
# Load dataset and build vocabularies
full_dataset = CarbonDataset(str(config.data_path), config=config)

# Override vocab sizes from actual data (may differ from config defaults)
config.vocab_materials = len(full_dataset.vocab["materials"]) + 1
config.vocab_steps = len(full_dataset.vocab["steps"]) + 1
config.vocab_categories = len(full_dataset.vocab["categories"]) + 1
config.vocab_subcategories = len(full_dataset.vocab["subcategories"]) + 1

print(f"Dataset: {len(full_dataset)} records")
print(f"Vocabs: materials={config.vocab_materials}, steps={config.vocab_steps}, "
      f"categories={config.vocab_categories}, subcategories={config.vocab_subcategories}")

# Stratified split
train_set, val_set, test_set = create_splits(
    full_dataset,
    category_names=full_dataset.category_names,
    n_materials_list=full_dataset.n_materials_list,
    seed=config.split_seed,
    train_ratio=config.split_ratios[0],
    val_ratio=config.split_ratios[1],
    test_ratio=config.split_ratios[2],
)
print(f"Split: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

# Fit transform on training targets
train_records = [full_dataset.records[i] for i in train_set.indices]
arrays = {h: np.array([r[f"cf_{h}"] for r in train_records]) for h in HEAD_NAMES}
transform = Log1pZScoreTransform().fit(
    arrays["raw_materials"], arrays["transport"],
    arrays["processing"], arrays["packaging"],
)
print(f"Transform fitted: {transform.state_dict()}")

# DataLoaders
train_loader = DataLoader(
    train_set, batch_size=config.batch_size, shuffle=True,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory and (device.type == "cuda"),
    persistent_workers=config.persistent_workers and (config.num_workers > 0),
)
val_loader = DataLoader(
    val_set, batch_size=config.batch_size, shuffle=False,
    num_workers=config.num_workers,
    pin_memory=config.pin_memory and (device.type == "cuda"),
    persistent_workers=config.persistent_workers and (config.num_workers > 0),
)
test_loader = DataLoader(
    test_set, batch_size=config.batch_size, shuffle=False,
)
print(f"Loaders: {len(train_loader)} train, {len(val_loader)} val, "
      f"{len(test_loader)} test batches")

## 7. Smoke Test

In [ ]:
# Run smoke test with a copy of config (smoke_test mutates it)
smoke_config = copy.deepcopy(config)
result = smoke_test(smoke_config)
assert result["passed"], f"Smoke test failed: {result}"
print(f"Smoke test passed -- train_loss={result['train_loss']:.4f}, "
      f"val_loss={result['val_loss']:.4f}, "
      f"grad_ratio={result['grad_ratio']:.2f}, "
      f"n_params={result['n_params']}")

## 8. Check for Existing Checkpoint

In [ ]:
checkpoint_pattern = os.path.join(config.checkpoint_dir, "checkpoint_*.pt")
existing = sorted(glob.glob(checkpoint_pattern))

if existing:
    print(f"Found {len(existing)} existing checkpoint(s):")
    for cp in existing:
        print(f"  {cp}")
    RESUME_FROM = existing[-1]
    print(f"Will resume from: {RESUME_FROM}")
else:
    RESUME_FROM = None
    print("No existing checkpoints. Training from scratch.")

## 9. Build Model and Trainer

In [ ]:
model = CarbonModel(config).to(device)
loss_fn = ThreeGroupLoss(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} parameters on {device}")

trainer = CarbonTrainer(
    config=config,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    val_loader=val_loader,
    transform=transform,
    device=device,
)

# Resume if checkpoint exists
if RESUME_FROM:
    resumed_epoch = trainer.load_checkpoint(RESUME_FROM)
    print(f"Resumed from epoch {resumed_epoch}")

## 10. Viability Check

In [ ]:
viable = trainer.viability_check(n_canary=config.canary_epochs)
assert viable, "Viability check failed -- see diagnostics above"
print("Viability check passed. Starting full training...")

## 11. Training

In [ ]:
best_metrics, history = trainer.train(
    max_epochs=config.max_epochs,
    patience=config.patience,
)
print(f"Training complete. Best val loss: {best_metrics.get('val_loss', 'N/A')}")

## 12. Evaluation

In [ ]:
# Load best checkpoint
best_ckpt_path = os.path.join(config.checkpoint_dir, "checkpoint_best.pt")
if os.path.exists(best_ckpt_path):
    trainer.load_checkpoint(best_ckpt_path)
    print(f"Loaded best checkpoint: {best_ckpt_path}")
else:
    print("No best checkpoint found; evaluating current model state")

# Test set evaluation (Tier E -- full data, no privileged)
print("\n--- Test Set (Tier E) ---")
test_loss, test_metrics = trainer.val_epoch(tier="E", loader=test_loader)
for k, v in sorted(test_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Per-tier evaluation matrix
print("\n--- Per-Tier Evaluation (Validation Set) ---")
tier_matrix = per_tier_evaluation(
    model=model,
    dataloader=val_loader,
    transform=transform,
    tiers=["A", "B", "C", "D", "E", "F"],
    device=device,
)
print(tier_matrix.to_string())

## 13. Visualizations

In [ ]:
setup_style()
PLOT_DIR = os.path.join(config.checkpoint_dir, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("Generating training curves...")
plot_training_curves(history, save_path=os.path.join(PLOT_DIR, "training_curves.png"))

print("Generating prediction scatter plots...")
plot_predictions(model, test_loader, transform, device,
                 save_path=os.path.join(PLOT_DIR, "predictions.png"))

print("Generating residual diagnostics...")
plot_residuals(model, test_loader, transform, device,
               save_path=os.path.join(PLOT_DIR, "residuals.png"))

print("Generating tier degradation curves...")
plot_tier_degradation(tier_matrix, save_path=os.path.join(PLOT_DIR, "tier_degradation.png"))

print("Generating tier MAE heatmap...")
plot_tier_heatmap(tier_matrix, save_path=os.path.join(PLOT_DIR, "tier_heatmap.png"))

print("Generating loss group dynamics...")
plot_loss_groups(history, save_path=os.path.join(PLOT_DIR, "loss_groups.png"))

print(f"\nAll plots saved to {PLOT_DIR}")

## 14. Summary

In [ ]:
print("=" * 60)
print("Carbon Footprint Model -- Training Summary")
print("=" * 60)
print(f"Parameters: {n_params:,}")
print(f"Device: {device}")
print(f"Epochs trained: {len(history)}")
if best_metrics:
    print(f"Best val loss: {best_metrics.get('val_loss', 'N/A'):.4f}")
    for head in HEAD_NAMES:
        mae_key = f"{head}_mae"
        r2_key = f"{head}_r2"
        if mae_key in best_metrics:
            print(f"  {head}: MAE={best_metrics[mae_key]:.4f}, "
                  f"R2={best_metrics.get(r2_key, 0.0):.4f}")
    if "total_mae" in best_metrics:
        print(f"  total: MAE={best_metrics['total_mae']:.4f}, "
              f"R2={best_metrics.get('total_r2', 0.0):.4f}")
print(f"Checkpoint: {config.checkpoint_dir}")
print(f"Plots: {PLOT_DIR}")
print(f"Experiment log: {os.path.join(config.checkpoint_dir, config.runs_log)}")